In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Inicializar SparkSession
spark = SparkSession.builder.appName("DataFrameCreation").getOrCreate()

# Crear un DataFrame de ejemplo 'df' con id_usuario, ventas y ciudad
data = [
    (1, 100, "Madrid"),
    (2, 150, "Barcelona"),
    (3, 200, "Valencia"),
    (4, 50, "Sevilla"),
    (5, 120, "Madrid"),
    (6, 180, "Bilbao"),
    (7, 200, "Valencia"),
    (8, 75, "Sevilla"),
    (9, 130, "Barcelona"),
    (10, 110, "Madrid")
]
columns = ["id_usuario", "ventas", "ciudad"]
df = spark.createDataFrame(data, columns)

# También creamos una vista temporal global ficticia para poder usar spark.sql posteriormente
df.createOrReplaceTempView("tabla_ventas")

print("DataFrame 'df' y vista temporal 'tabla_ventas' creados exitosamente.")
print("Esquema de df:")
df.printSchema()
print("Datos de muestra de df:")
df.show(5)

DataFrame 'df' y vista temporal 'tabla_ventas' creados exitosamente.
Esquema de df:
root
 |-- id_usuario: long (nullable = true)
 |-- ventas: long (nullable = true)
 |-- ciudad: string (nullable = true)

Datos de muestra de df:
+----------+------+---------+
|id_usuario|ventas|   ciudad|
+----------+------+---------+
|         1|   100|   Madrid|
|         2|   150|Barcelona|
|         3|   200| Valencia|
|         4|    50|  Sevilla|
|         5|   120|   Madrid|
+----------+------+---------+
only showing top 5 rows


### Selección y derivación de datos a nivel de columna

In [4]:
from pyspark.sql.functions import col, round

# 1. DataFrame API
df.select(
    col("id_usuario"), col("ventas")
    ).withColumn( "con_iva", round(col("ventas") * 1.21, 2)).show()

+----------+------+-------+
|id_usuario|ventas|con_iva|
+----------+------+-------+
|         1|   100|  121.0|
|         2|   150|  181.5|
|         3|   200|  242.0|
|         4|    50|   60.5|
|         5|   120|  145.2|
|         6|   180|  217.8|
|         7|   200|  242.0|
|         8|    75|  90.75|
|         9|   130|  157.3|
|        10|   110|  133.1|
+----------+------+-------+



In [5]:
# 2. Spark SQL
spark.sql("""
  SELECT id_usuario, ventas,
  (ventas * 1.21)AS con_iva
  FROM tabla_ventas """).show()

+----------+------+-------+
|id_usuario|ventas|con_iva|
+----------+------+-------+
|         1|   100| 121.00|
|         2|   150| 181.50|
|         3|   200| 242.00|
|         4|    50|  60.50|
|         5|   120| 145.20|
|         6|   180| 217.80|
|         7|   200| 242.00|
|         8|    75|  90.75|
|         9|   130| 157.30|
|        10|   110| 133.10|
+----------+------+-------+



### Filtrado de datos a nivel de filas

In [9]:
# 1. DataFrame API: Usando col() para referenciar la columna
from pyspark.sql.functions import col
df_filtrado = df.filter(col("ventas") > 120)

df_filtrado.show()

+----------+------+---------+
|id_usuario|ventas|   ciudad|
+----------+------+---------+
|         2|   150|Barcelona|
|         3|   200| Valencia|
|         6|   180|   Bilbao|
|         7|   200| Valencia|
|         9|   130|Barcelona|
+----------+------+---------+



In [10]:
# 2. Spark SQL Nativo: la cláusula WHERE clásica de SQL
df_filtrado_sql = spark.sql("""
    SELECT *
    FROM tabla_ventas
    WHERE ventas > 120
""")

df_filtrado.show(3)

+----------+------+---------+
|id_usuario|ventas|   ciudad|
+----------+------+---------+
|         2|   150|Barcelona|
|         3|   200| Valencia|
|         6|   180|   Bilbao|
+----------+------+---------+
only showing top 3 rows


### Agrupación de datos

In [11]:
# 1. DataFrame API: Agrupar por ciudad y sumar la columna ventas
df_agrupado = df.groupBy("ciudad").sum("ventas")

df_agrupado.show()

+---------+-----------+
|   ciudad|sum(ventas)|
+---------+-----------+
|   Madrid|        330|
|Barcelona|        280|
| Valencia|        400|
|  Sevilla|        125|
|   Bilbao|        180|
+---------+-----------+



In [ ]:
# 2. Spark SQL Nativo: Usar GROUP BY y la función de agregación SUM()
df_agrupado_sql = spark.sql("""
    SELECT ciudad, SUM(ventas) as total_ventas
    FROM tabla_ventas
    GROUP BY ciudad
""")

df_agrupado.show()

### Cruces de datos (JOIN)

In [13]:
# Preparación rápida de una tabla adicional de zonas libres/gratuitas de envío.
df_zonas = spark.createDataFrame([("Madrid", "Centro"), ("Barcelona", "Este")], ["ciudad", "zona"])
df_zonas.createOrReplaceTempView("tabla_zonas")
df_zonas.show()

+---------+------+
|   ciudad|  zona|
+---------+------+
|   Madrid|Centro|
|Barcelona|  Este|
+---------+------+



In [20]:
# 1. DataFrame API: cruzamos usando la columna común "ciudad"
df_join = df.join(df_zonas, on="ciudad", how="inner")

df_join.show()

+---------+----------+------+------+
|   ciudad|id_usuario|ventas|  zona|
+---------+----------+------+------+
|Barcelona|         2|   150|  Este|
|Barcelona|         9|   130|  Este|
|   Madrid|         1|   100|Centro|
|   Madrid|         5|   120|Centro|
|   Madrid|        10|   110|Centro|
+---------+----------+------+------+



In [16]:
# 2. Spark SQL Nativo: Usando INNER JOIN estándar
df_join_sql = spark.sql("""
    SELECT v.id_usuario, v.ventas, v.ciudad, z.zona
    FROM tabla_ventas v
    INNER JOIN tabla_zonas z
    ON v.ciudad = z.ciudad
""")

df_join.show()

+---------+----------+------+------+
|   ciudad|id_usuario|ventas|  zona|
+---------+----------+------+------+
|Barcelona|         2|   150|  Este|
|Barcelona|         9|   130|  Este|
|   Madrid|         1|   100|Centro|
|   Madrid|         5|   120|Centro|
|   Madrid|        10|   110|Centro|
+---------+----------+------+------+

